# 📍 Model 02 — Station Day AQI Prediction
**AirVintage Production ML Pipeline — Deployment-Ready**

| Input | Source | Free? |
|-------|--------|-------|
| `PM2.5`, `PM10` | OpenAQ / CPCB API | ✅ |
| `NO2`, `SO2`, `CO`, `O3` | OpenAQ / CPCB API | ✅ |
| `Station ID` | CPCB station list | ✅ |
| `Date` | System clock | ✅ |

**Dropped:** `NO`, `NOx`, `NH3`, `Benzene`, `Toluene`, `Xylene`

In [ ]:
import sys, os, warnings, json
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.'))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score

from airvintage_ml import (
    aqi_to_bucket, health_advisory,
    add_temporal_features, encode_categorical,
    fillna_production, add_entity_stats,
    compute_metrics, print_metrics_table,
    plot_actual_vs_pred, plot_residuals,
    plot_feature_importance, plot_learning_curve_lgb,
    update_model_registry,
)

DATASET='station_day_cleaned.csv'; MODEL_KEY='station_day'
MODELS_DIR='models'; REGISTRY=f'{MODELS_DIR}/model_registry.json'; SEED=42
os.makedirs(MODELS_DIR, exist_ok=True)

CORE_POLLUTANTS = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3']
print(f'LightGBM {lgb.__version__} | Inputs: {CORE_POLLUTANTS}')

In [ ]:
df = pd.read_csv(DATASET, parse_dates=['Date'])
print(f'Shape: {df.shape} | Stations: {df["StationId"].nunique()}')
df.head(3)

In [ ]:
# ── EDA
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].hist(df['AQI'].dropna(), bins=60, color='#f093fb', edgecolor='white', alpha=0.85)
axes[0,0].set_title('AQI Distribution', fontweight='bold')

bc = df['AQI_Bucket'].value_counts()
axes[0,1].pie(bc.values, labels=bc.index, autopct='%1.1f%%',
              colors=['#2ecc71','#f1c40f','#e67e22','#e74c3c','#8e44ad','#2c3e50'][:len(bc)])
axes[0,1].set_title('AQI Bucket Split', fontweight='bold')

avail = [c for c in CORE_POLLUTANTS if c in df.columns]
corr = df[avail + ['AQI']].corr()['AQI'].drop('AQI').abs().sort_values()
axes[0,2].barh(corr.index, corr.values, color='#667eea')
axes[0,2].set_title('Core Pollutant Correlation with AQI', fontweight='bold')

stn_mean = df.groupby('StationId')['AQI'].mean()
top = stn_mean.sort_values(ascending=False).head(10)
axes[1,0].barh(top.index[::-1], top.values[::-1], color=plt.cm.Reds(np.linspace(0.4,0.9,10)))
axes[1,0].set_title('Top 10 Polluted Stations', fontweight='bold')
axes[1,0].tick_params(axis='y', labelsize=7)

axes[1,1].hist(stn_mean, bins=40, color='#4facfe', edgecolor='white', alpha=0.85)
axes[1,1].set_title('Station Mean AQI Distribution', fontweight='bold')

df['Month_tmp'] = df['Date'].dt.month
monthly = df.groupby('Month_tmp')['AQI'].mean()
axes[1,2].plot(range(1,13), monthly.values, 'o-', color='#f12711', lw=2.5)
axes[1,2].set_xticks(range(1,13))
axes[1,2].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[1,2].set_title('Monthly Avg AQI', fontweight='bold')

fig.suptitle('Station Day EDA — Production Feature Set', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/eda_02_station_day.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Engineering — Production Only
df_feat = df.copy()
df_feat = add_temporal_features(df_feat, 'Date')

# Computed interactions (no extra API needed)
df_feat['PM_ratio'] = df_feat['PM2.5'] / (df_feat['PM10'] + 1e-6)
df_feat['Total_PM'] = df_feat['PM2.5'] + df_feat['PM10']
df_feat['NO2_SO2']  = df_feat['NO2']   + df_feat['SO2']
df_feat['CO_O3']    = df_feat['CO']    * df_feat['O3']

df_feat, le_stn  = encode_categorical(df_feat, 'StationId')
df_feat, stn_stats = add_entity_stats(df_feat, 'StationId_encoded', 'AQI')

DROP = ['StationId','Date','AQI','AQI_Bucket','Month_tmp',
        'NO','NOx','NH3','Benzene','Toluene','Xylene']
FEATURE_COLS = [c for c in df_feat.columns if c not in DROP]

df_clean = df_feat.dropna(subset=['AQI']).copy()
df_clean = fillna_production(df_clean)
X, y = df_clean[FEATURE_COLS], df_clean['AQI']
print(f'Production features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

In [ ]:
X_train,X_tmp,y_train,y_tmp = train_test_split(X,y,test_size=0.30,random_state=SEED)
X_val,X_test,y_val,y_test   = train_test_split(X_tmp,y_tmp,test_size=0.50,random_state=SEED)
print(f'Train:{X_train.shape[0]:,} | Val:{X_val.shape[0]:,} | Test:{X_test.shape[0]:,}')

In [ ]:
# ── Random Forest
rf = RandomForestRegressor(n_estimators=400, max_depth=22, min_samples_leaf=2,
                            max_features=0.7, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
rf_r2 = compute_metrics(y_val, rf.predict(X_val))['R2']
print(f'RF Val R2: {rf_r2:.4f}')

In [ ]:
# ── LightGBM
lgb_m = lgb.LGBMRegressor(
    n_estimators=1200, learning_rate=0.04, max_depth=9,
    num_leaves=127, min_child_samples=20, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_m.fit(X_train, y_train, eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(60, verbose=False), lgb.log_evaluation(200)])
lgb_r2 = compute_metrics(y_val, lgb_m.predict(X_val))['R2']
print(f'LGB Val R2: {lgb_r2:.4f}')

In [ ]:
# ── Weighted Ensemble
total = rf_r2 + lgb_r2
w_rf, w_lgb = rf_r2/total, lgb_r2/total
ens_pred = lambda X: w_rf * rf.predict(X) + w_lgb * lgb_m.predict(X)

test_m_rf  = compute_metrics(y_test, rf.predict(X_test))
test_m_lgb = compute_metrics(y_test, lgb_m.predict(X_test))
test_m_ens = compute_metrics(y_test, ens_pred(X_test))
print_metrics_table({'RF':test_m_rf,'LGB':test_m_lgb,'Ensemble':test_m_ens})

plot_actual_vs_pred(y_test, ens_pred(X_test), 'Actual vs Pred — Station Day', f'{MODELS_DIR}/avp_02_station_day.png')
plot_residuals(y_test, ens_pred(X_test), 'Residuals — Station Day', f'{MODELS_DIR}/res_02_station_day.png')
plot_feature_importance(FEATURE_COLS, lgb_m.feature_importances_, 'Feature Importance — Station Day', savepath=f'{MODELS_DIR}/fi_02_station_day.png')
plot_learning_curve_lgb(lgb_m, 'LGB Curve — Station Day', f'{MODELS_DIR}/lc_02_station_day.png')

In [ ]:
joblib.dump(rf,    f'{MODELS_DIR}/02_station_day_rf.pkl')
lgb_m.booster_.save_model(f'{MODELS_DIR}/02_station_day_lgb.txt')
joblib.dump(le_stn, f'{MODELS_DIR}/02_station_day_le_station.pkl')
stn_stats.to_csv(f'{MODELS_DIR}/02_station_day_stn_stats.csv', index=False)
with open(f'{MODELS_DIR}/02_station_day_ensemble_weights.json','w') as f:
    json.dump({'w_rf':w_rf,'w_lgb':w_lgb},f,indent=2)
with open(f'{MODELS_DIR}/02_station_day_features.json','w') as f:
    json.dump(FEATURE_COLS,f,indent=2)
with open(f'{MODELS_DIR}/02_station_day_input_schema.json','w') as f:
    json.dump({'required_inputs':['station_id','date']+CORE_POLLUTANTS,
               'note':'All from OpenAQ/CPCB free API'},f,indent=2)

update_model_registry(REGISTRY, MODEL_KEY, 'Weighted Ensemble (RF+LGB)', DATASET, test_m_ens,
    model_paths={'rf':f'{MODELS_DIR}/02_station_day_rf.pkl','lgb':f'{MODELS_DIR}/02_station_day_lgb.txt',
                 'encoder':f'{MODELS_DIR}/02_station_day_le_station.pkl','stats':f'{MODELS_DIR}/02_station_day_stn_stats.csv',
                 'weights':f'{MODELS_DIR}/02_station_day_ensemble_weights.json','features':f'{MODELS_DIR}/02_station_day_features.json'},
    feature_count=len(FEATURE_COLS), notes='Inputs: PM2.5,PM10,NO2,SO2,CO,O3 + station_id + date')
print('Model 02 saved.')

In [ ]:
def predict_station_day(
    station_id: str, date: str,
    pm25: float, pm10: float,
    no2: float, so2: float, co: float, o3: float
) -> dict:
    """
    Predict daily AQI for a monitoring station.
    All inputs available from CPCB / OpenAQ free API.
    """
    row = pd.DataFrame([{'StationId':station_id,'Date':date,
                          'PM2.5':pm25,'PM10':pm10,'NO2':no2,'SO2':so2,'CO':co,'O3':o3}])
    row['Date'] = pd.to_datetime(row['Date'])
    row = add_temporal_features(row, 'Date')
    row['PM_ratio']=pm25/(pm10+1e-6); row['Total_PM']=pm25+pm10
    row['NO2_SO2']=no2+so2; row['CO_O3']=co*o3

    s_enc = int(le_stn.transform([station_id])[0]) if station_id in le_stn.classes_ else -1
    row['StationId_encoded'] = s_enc
    sr = stn_stats[stn_stats['StationId_encoded']==s_enc]
    for col in [c for c in stn_stats.columns if 'AQI' in c]:
        row[col] = float(sr[col].values[0]) if len(sr)>0 else 150.0

    X_in = row[FEATURE_COLS].fillna(0)
    aqi  = float(w_rf*rf.predict(X_in)[0] + w_lgb*lgb_m.predict(X_in)[0])
    b    = aqi_to_bucket(aqi)
    return {'model':'station_day','station_id':station_id,'date':date,
            'predicted_aqi':round(aqi,2),'aqi_category':b,'health_advisory':health_advisory(b)}

demo = predict_station_day(df['StationId'].iloc[0], '2025-06-10',
                            pm25=60, pm10=90, no2=35, so2=8, co=0.8, o3=25)
print('Demo:'); [print(f'  {k}: {v}') for k,v in demo.items()]